# Script-Adversarial Writer Metric Learning

This notebook investigates whether explicit script suppression can reduce script accessibility in writer embeddings while preserving writer-verification utility.

## Motivation

The current best SSL-initialized dynamic metric representation achieves strong writer-verification performance, but script remains almost perfectly linearly recoverable.

Writer-disjoint validation results from the previous experiments include:

- Hybrid writer-verification ROC-AUC: 0.7441
- Hybrid script linear-probe ROC-AUC: 0.9999
- Hybrid minimum stratified script ROC-AUC: 0.9821

Therefore, improved writer verification has not produced script invariance.

## Research question

Can an explicit adversarial script objective reduce script information in the embedding without destroying writer discrimination?

## Controlled starting point

The experiment starts from the same self-supervised epoch-20 initialization used by the previous SSL-initialized dynamic metric-learning experiment.

The writer metric-learning protocol remains unchanged:

- 226 development-training writers
- Arabic page 1 and page 2 writer pairs
- 226 genuine pairs per epoch
- 226 impostor pairs per epoch
- deterministic dynamic negative-writer sampling
- 10 distinct negative writers per anchor across 10 epochs
- ResNet-18
- layer 4 trainable
- earlier backbone frozen
- BatchNorm running statistics frozen
- 512-dimensional L2-normalized embedding
- CosineEmbeddingLoss with margin 0.5
- validation-only checkpoint selection

## Script-adversarial branch

The writer-pair batches contain only Arabic handwriting and therefore cannot provide a meaningful script-classification objective.

A separate script batch stream will consequently use all 904 development-training images:

- 452 Arabic images
- 452 English images
- the same 226 development writers
- no validation writers
- no official-test writers

A script classifier will receive the 512-dimensional writer embedding through a gradient-reversal layer.

The script classifier is optimized to predict Arabic versus English.

The encoder receives the reversed script-classification gradient and is therefore encouraged to make script prediction difficult while continuing to optimize writer metric learning.

## Evaluation principles

Writer utility and nuisance suppression will be evaluated separately.

Writer utility:

- validation writer-verification ROC-AUC
- validation EER
- within-script macro ROC-AUC
- cross-script macro ROC-AUC
- within-minus-cross-script gap

Script accessibility:

- writer-disjoint validation linear-probe accuracy
- writer-disjoint validation script ROC-AUC

A decrease in script-probe performance will not automatically be considered successful.

A useful invariant representation must reduce script accessibility while preserving meaningful writer-verification performance.

The first stage of this notebook will establish the architecture, gradient behavior, and zero-adversarial control before evaluating non-zero adversarial strength.

The official-test split will not be used to select adversarial strength, checkpoint, or training configuration.

In [1]:
import copy
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.autograd import Function
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18

from handwriting_cross_script_research import (
    QUWIDataset,
    load_image_tensor,
)

In [2]:
SEED = 42
EPOCHS = 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    PROJECT_ROOT / "pyproject.toml"
).exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (
    PROJECT_ROOT
    / "pyproject.toml"
).exists()

IMAGE_DIR = (
    Path.home()
    / "Documents"
    / "Handwriting"
    / "QUWI"
    / "extracted"
    / "images"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

VALIDATION_PAIR_PATH = (
    PROJECT_ROOT
    / "splits"
    / "verification"
    / "quwi_validation_verification_pairs.csv"
)

TEST_PAIR_PATH = (
    PROJECT_ROOT
    / "splits"
    / "verification"
    / "quwi_official_test_verification_pairs.csv"
)

SSL_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "self_supervised_pretraining"
    / "resnet18_moco_ssl_best.pt"
)

HYBRID_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "ssl_initialized_metric_learning"
    / "resnet18_ssl_initialized_dynamic_metric_best.pt"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "script_adversarial_metric_learning"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "script_adversarial_metric_learning"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

split_df = pd.read_csv(
    SPLIT_PATH
)

development_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "development_train"
    ]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "validation"
    ]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "official_test"
    ]
    .copy()
    .reset_index(drop=True)
)

validation_pairs_df = pd.read_csv(
    VALIDATION_PAIR_PATH
)

print(
    "Development images:",
    len(
        development_df
    ),
)

print(
    "Development writers:",
    development_df[
        "writer"
    ].nunique(),
)

print(
    "Arabic images:",
    int(
        (
            development_df[
                "language"
            ] == "Arabic"
        ).sum()
    ),
)

print(
    "English images:",
    int(
        (
            development_df[
                "language"
            ] == "English"
        ).sum()
    ),
)

print(
    "Validation writers:",
    validation_df[
        "writer"
    ].nunique(),
)

print(
    "Validation pairs:",
    len(
        validation_pairs_df
    ),
)

print(
    "SSL checkpoint exists:",
    SSL_CHECKPOINT_PATH.exists(),
)

print(
    "Hybrid baseline checkpoint exists:",
    HYBRID_CHECKPOINT_PATH.exists(),
)

print(
    "Device:",
    DEVICE,
)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        ),
    )

assert len(
    development_df
) == 904

assert development_df[
    "writer"
].nunique() == 226

assert (
    development_df[
        "language"
    ] == "Arabic"
).sum() == 452

assert (
    development_df[
        "language"
    ] == "English"
).sum() == 452

assert validation_df[
    "writer"
].nunique() == 56

assert len(
    validation_pairs_df
) == 18_816

assert SSL_CHECKPOINT_PATH.exists()

assert HYBRID_CHECKPOINT_PATH.exists()

Development images: 904
Development writers: 226
Arabic images: 452
English images: 452
Validation writers: 56
Validation pairs: 18816
SSL checkpoint exists: True
Hybrid baseline checkpoint exists: True
Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
page1_df = (
    development_df[
        development_df[
            "page_id"
        ] == 1
    ]
    .sort_values(
        "writer"
    )
    .reset_index(drop=True)
)

page2_df = (
    development_df[
        development_df[
            "page_id"
        ] == 2
    ]
    .sort_values(
        "writer"
    )
    .reset_index(drop=True)
)

writers = page1_df[
    "writer"
].to_numpy()

assert np.array_equal(
    writers,
    page2_df[
        "writer"
    ].to_numpy(),
)

page1_filename = dict(
    zip(
        page1_df[
            "writer"
        ],
        page1_df[
            "filename"
        ],
    )
)

page2_filename = dict(
    zip(
        page2_df[
            "writer"
        ],
        page2_df[
            "filename"
        ],
    )
)

offset_rng = np.random.default_rng(
    SEED
)

epoch_offsets = offset_rng.choice(
    np.arange(
        1,
        len(
            writers
        ),
    ),
    size=EPOCHS,
    replace=False,
)

schedule_rows = []

for epoch, offset in enumerate(
    epoch_offsets,
    start=1,
):
    negative_indices = np.roll(
        np.arange(
            len(
                writers
            )
        ),
        -int(
            offset
        ),
    )

    epoch_rows = []

    for anchor_index, anchor_writer in enumerate(
        writers
    ):
        negative_writer = writers[
            negative_indices[
                anchor_index
            ]
        ]

        epoch_rows.append(
            {
                "epoch": epoch,
                "offset": int(
                    offset
                ),
                "anchor_writer": int(
                    anchor_writer
                ),
                "partner_writer": int(
                    anchor_writer
                ),
                "filename_a": page1_filename[
                    anchor_writer
                ],
                "filename_b": page2_filename[
                    anchor_writer
                ],
                "pair_label": 1,
            }
        )

        epoch_rows.append(
            {
                "epoch": epoch,
                "offset": int(
                    offset
                ),
                "anchor_writer": int(
                    anchor_writer
                ),
                "partner_writer": int(
                    negative_writer
                ),
                "filename_a": page1_filename[
                    anchor_writer
                ],
                "filename_b": page2_filename[
                    negative_writer
                ],
                "pair_label": 0,
            }
        )

    schedule_rows.append(
        pd.DataFrame(
            epoch_rows
        ).sample(
            frac=1.0,
            random_state=(
                SEED
                + epoch
            ),
        )
    )

dynamic_training_pairs_df = pd.concat(
    schedule_rows,
    ignore_index=True,
)


class QUWIPairDataset(Dataset):
    def __init__(
        self,
        pair_df,
        image_dir,
    ):
        self.pair_df = (
            pair_df
            .reset_index(drop=True)
            .copy()
        )

        self.image_dir = Path(
            image_dir
        )

    def __len__(self):
        return len(
            self.pair_df
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.pair_df.iloc[
            index
        ]

        image_a, _ = load_image_tensor(
            self.image_dir
            / row[
                "filename_a"
            ]
        )

        image_b, _ = load_image_tensor(
            self.image_dir
            / row[
                "filename_b"
            ]
        )

        return {
            "image_a": image_a,
            "image_b": image_b,
            "pair_label": torch.tensor(
                row[
                    "pair_label"
                ],
                dtype=torch.float32,
            ),
        }


script_dataset = QUWIDataset(
    metadata=development_df,
    image_dir=IMAGE_DIR,
)


def create_metric_loader(
    epoch,
):
    epoch_pairs_df = (
        dynamic_training_pairs_df[
            dynamic_training_pairs_df[
                "epoch"
            ] == epoch
        ]
        .reset_index(drop=True)
    )

    pair_dataset = QUWIPairDataset(
        epoch_pairs_df,
        IMAGE_DIR,
    )

    generator = (
        torch.Generator()
        .manual_seed(
            SEED
            + epoch
        )
    )

    loader = DataLoader(
        pair_dataset,
        batch_size=8,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        generator=generator,
    )

    return (
        epoch_pairs_df,
        loader,
    )


def create_script_loader(
    epoch,
):
    generator = (
        torch.Generator()
        .manual_seed(
            SEED
            + 10_000
            + epoch
        )
    )

    return DataLoader(
        script_dataset,
        batch_size=16,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        generator=generator,
    )


epoch1_pairs_df, epoch1_metric_loader = (
    create_metric_loader(
        1
    )
)

epoch1_script_loader = (
    create_script_loader(
        1
    )
)

metric_batch = next(
    iter(
        epoch1_metric_loader
    )
)

script_batch = next(
    iter(
        epoch1_script_loader
    )
)

script_unique_labels, script_unique_counts = (
    torch.unique(
        script_batch[
            "language_label"
        ],
        return_counts=True,
    )
)

script_batch_distribution = {
    int(
        label
    ): int(
        count
    )
    for label, count in zip(
        script_unique_labels,
        script_unique_counts,
    )
}

print(
    "Epoch offsets:",
    epoch_offsets.tolist(),
)

print(
    "Metric pairs per epoch:",
    len(
        epoch1_pairs_df
    ),
)

print(
    "Metric batches per epoch:",
    len(
        epoch1_metric_loader
    ),
)

print(
    "Script images per epoch:",
    len(
        script_dataset
    ),
)

print(
    "Script batches per epoch:",
    len(
        epoch1_script_loader
    ),
)

print(
    "Metric batch shape:",
    metric_batch[
        "image_a"
    ].shape,
)

print(
    "Script batch shape:",
    script_batch[
        "image"
    ].shape,
)

print(
    "Script labels in first batch:",
    script_batch_distribution,
)

print(
    "Total Arabic script samples:",
    int(
        (
            development_df[
                "language"
            ] == "Arabic"
        ).sum()
    ),
)

print(
    "Total English script samples:",
    int(
        (
            development_df[
                "language"
            ] == "English"
        ).sum()
    ),
)

assert epoch_offsets.tolist() == [
    222,
    168,
    20,
    143,
    97,
    96,
    156,
    22,
    46,
    190,
]

assert len(
    epoch1_pairs_df
) == 452

assert len(
    epoch1_metric_loader
) == 57

assert len(
    script_dataset
) == 904

assert len(
    epoch1_script_loader
) == 57

assert metric_batch[
    "image_a"
].shape == (
    8,
    3,
    384,
    384,
)

assert script_batch[
    "image"
].shape == (
    16,
    3,
    384,
    384,
)

Epoch offsets: [222, 168, 20, 143, 97, 96, 156, 22, 46, 190]
Metric pairs per epoch: 452
Metric batches per epoch: 57
Script images per epoch: 904
Script batches per epoch: 57
Metric batch shape: torch.Size([8, 3, 384, 384])
Script batch shape: torch.Size([16, 3, 384, 384])
Script labels in first batch: {0: 8, 1: 8}
Total Arabic script samples: 452
Total English script samples: 452


In [4]:
import gc

gc.collect()

CUDA_HEALTHY = False
CUDA_ERROR = None

if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()

        test_tensor = torch.zeros(
            (
                64,
                64,
            ),
            device="cuda",
        )

        test_result = (
            test_tensor
            + 1
        ).sum()

        CUDA_HEALTHY = bool(
            torch.isfinite(
                test_result
            ).item()
        )

        del test_tensor
        del test_result

        torch.cuda.empty_cache()

    except Exception as error:
        CUDA_ERROR = str(
            error
        )

print(
    "torch.cuda.is_available():",
    torch.cuda.is_available(),
)

print(
    "CUDA healthy:",
    CUDA_HEALTHY,
)

if CUDA_ERROR is not None:
    print(
        "CUDA error:",
        CUDA_ERROR.split(
            "\n"
        )[0],
    )

if CUDA_HEALTHY:
    TRAIN_DEVICE = torch.device(
        "cuda"
    )
else:
    TRAIN_DEVICE = torch.device(
        "cpu"
    )

print(
    "Training device candidate:",
    TRAIN_DEVICE,
)

torch.cuda.is_available(): True
CUDA healthy: True
Training device candidate: cuda


In [5]:
IMAGENET_MEAN = (
    0.485,
    0.456,
    0.406,
)

IMAGENET_STD = (
    0.229,
    0.224,
    0.225,
)


class GradientReversalFunction(Function):
    @staticmethod
    def forward(
        ctx,
        inputs,
        coefficient,
    ):
        ctx.coefficient = float(
            coefficient
        )

        return inputs.view_as(
            inputs
        )

    @staticmethod
    def backward(
        ctx,
        grad_output,
    ):
        return (
            -ctx.coefficient
            * grad_output,
            None,
        )


def gradient_reverse(
    inputs,
    coefficient,
):
    return GradientReversalFunction.apply(
        inputs,
        coefficient,
    )


class ScriptAdversarialWriterModel(nn.Module):
    def __init__(
        self,
    ):
        super().__init__()

        backbone = resnet18(
            weights=None
        )

        embedding_dim = (
            backbone.fc.in_features
        )

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.script_classifier = nn.Linear(
            embedding_dim,
            2,
        )

        self.register_buffer(
            "normalization_mean",
            torch.tensor(
                IMAGENET_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                3,
                1,
                1,
            ),
        )

        self.register_buffer(
            "normalization_std",
            torch.tensor(
                IMAGENET_STD,
                dtype=torch.float32,
            ).view(
                1,
                3,
                1,
                1,
            ),
        )

        for parameter in (
            self.backbone.parameters()
        ):
            parameter.requires_grad = False

        for parameter in (
            self.backbone
            .layer4
            .parameters()
        ):
            parameter.requires_grad = True

    def train(
        self,
        mode=True,
    ):
        super().train(
            mode
        )

        self.backbone.eval()

        self.script_classifier.train(
            mode
        )

        return self

    def encode(
        self,
        images,
    ):
        normalized_images = (
            images
            - self.normalization_mean
        ) / self.normalization_std

        embeddings = self.backbone(
            normalized_images
        )

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1,
        )

        return embeddings

    def classify_script(
        self,
        embeddings,
        grl_coefficient,
    ):
        reversed_embeddings = gradient_reverse(
            embeddings,
            grl_coefficient,
        )

        return self.script_classifier(
            reversed_embeddings
        )

    def forward(
        self,
        images,
        grl_coefficient=1.0,
    ):
        embeddings = self.encode(
            images
        )

        script_logits = self.classify_script(
            embeddings,
            grl_coefficient,
        )

        return (
            embeddings,
            script_logits,
        )


ssl_checkpoint = torch.load(
    SSL_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

ssl_state_dict = ssl_checkpoint[
    "model_state_dict"
]

ssl_backbone_prefix = (
    "encoder_q.backbone."
)

ssl_backbone_state_dict = {
    key[
        len(
            ssl_backbone_prefix
        ):
    ]: value
    for key, value in (
        ssl_state_dict.items()
    )
    if key.startswith(
        ssl_backbone_prefix
    )
}

print(
    "SSL checkpoint epoch:",
    ssl_checkpoint[
        "epoch"
    ],
)

print(
    "Extracted SSL backbone tensors:",
    len(
        ssl_backbone_state_dict
    ),
)

assert ssl_checkpoint[
    "epoch"
] == 20

assert len(
    ssl_backbone_state_dict
) > 0

torch.manual_seed(
    SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        SEED
    )

adversarial_model = (
    ScriptAdversarialWriterModel()
    .to(
        TRAIN_DEVICE
    )
)

load_result = (
    adversarial_model
    .backbone
    .load_state_dict(
        ssl_backbone_state_dict,
        strict=True,
    )
)

adversarial_model.eval()

smoke_images = (
    script_batch[
        "image"
    ][:4]
    .to(
        TRAIN_DEVICE
    )
)

with torch.no_grad():
    (
        smoke_embeddings,
        smoke_script_logits,
    ) = adversarial_model(
        smoke_images,
        grl_coefficient=0.0,
    )

smoke_embedding_norms = (
    smoke_embeddings.norm(
        p=2,
        dim=1,
    )
)

early_backbone_trainable = any(
    parameter.requires_grad
    for parameter in (
        adversarial_model
        .backbone
        .layer3
        .parameters()
    )
)

layer4_trainable = any(
    parameter.requires_grad
    for parameter in (
        adversarial_model
        .backbone
        .layer4
        .parameters()
    )
)

script_classifier_trainable = all(
    parameter.requires_grad
    for parameter in (
        adversarial_model
        .script_classifier
        .parameters()
    )
)

print(
    "Missing backbone keys:",
    load_result.missing_keys,
)

print(
    "Unexpected backbone keys:",
    load_result.unexpected_keys,
)

print(
    "Embedding shape:",
    tuple(
        smoke_embeddings.shape
    ),
)

print(
    "Script-logit shape:",
    tuple(
        smoke_script_logits.shape
    ),
)

print(
    "Embedding norms:",
    smoke_embedding_norms
    .detach()
    .cpu()
    .numpy()
    .round(6)
    .tolist(),
)

print(
    "Earlier backbone trainable:",
    early_backbone_trainable,
)

print(
    "Layer 4 trainable:",
    layer4_trainable,
)

print(
    "Script classifier trainable:",
    script_classifier_trainable,
)

print(
    "Backbone training mode:",
    adversarial_model
    .backbone
    .training,
)

assert load_result.missing_keys == []

assert load_result.unexpected_keys == []

assert smoke_embeddings.shape == (
    4,
    512,
)

assert smoke_script_logits.shape == (
    4,
    2,
)

assert torch.allclose(
    smoke_embedding_norms,
    torch.ones_like(
        smoke_embedding_norms
    ),
    atol=1e-5,
)

assert early_backbone_trainable is False

assert layer4_trainable is True

assert script_classifier_trainable is True

assert (
    adversarial_model
    .backbone
    .training
    is False
)

SSL checkpoint epoch: 20
Extracted SSL backbone tensors: 120
Missing backbone keys: []
Unexpected backbone keys: []
Embedding shape: (4, 512)
Script-logit shape: (4, 2)
Embedding norms: [1.0, 1.0, 1.0, 1.0]
Earlier backbone trainable: False
Layer 4 trainable: True
Script classifier trainable: True
Backbone training mode: False


In [6]:
toy_inputs = torch.tensor(
    [
        [
            1.0,
            -2.0,
            0.5,
        ]
    ],
    dtype=torch.float32,
    requires_grad=True,
)

toy_weights = torch.tensor(
    [
        [
            0.30,
            -0.40,
            0.20,
        ]
    ],
    dtype=torch.float32,
)

baseline_output = (
    toy_inputs
    @ toy_weights.T
).sum()

baseline_output.backward()

baseline_gradient = (
    toy_inputs.grad
    .detach()
    .clone()
)

print(
    "Baseline forward value:",
    float(
        baseline_output.detach()
    ),
)

print(
    "Baseline gradient:",
    baseline_gradient.tolist(),
)

for coefficient in [
    0.0,
    0.5,
    1.0,
]:
    grl_inputs = (
        toy_inputs
        .detach()
        .clone()
        .requires_grad_(
            True
        )
    )

    reversed_inputs = gradient_reverse(
        grl_inputs,
        coefficient,
    )

    grl_output = (
        reversed_inputs
        @ toy_weights.T
    ).sum()

    grl_output.backward()

    grl_gradient = (
        grl_inputs.grad
        .detach()
        .clone()
    )

    expected_gradient = (
        -coefficient
        * baseline_gradient
    )

    print()

    print(
        "GRL coefficient:",
        coefficient,
    )

    print(
        "Forward value:",
        float(
            grl_output.detach()
        ),
    )

    print(
        "Observed gradient:",
        grl_gradient.tolist(),
    )

    print(
        "Expected gradient:",
        expected_gradient.tolist(),
    )

    assert torch.allclose(
        grl_output.detach(),
        baseline_output.detach(),
        atol=1e-7,
    )

    assert torch.allclose(
        grl_gradient,
        expected_gradient,
        atol=1e-7,
    )

print()
print(
    "Gradient-reversal sanity check passed."
)

Baseline forward value: 1.2000000476837158
Baseline gradient: [[0.30000001192092896, -0.4000000059604645, 0.20000000298023224]]

GRL coefficient: 0.0
Forward value: 1.2000000476837158
Observed gradient: [[-0.0, 0.0, -0.0]]
Expected gradient: [[-0.0, 0.0, -0.0]]

GRL coefficient: 0.5
Forward value: 1.2000000476837158
Observed gradient: [[-0.15000000596046448, 0.20000000298023224, -0.10000000149011612]]
Expected gradient: [[-0.15000000596046448, 0.20000000298023224, -0.10000000149011612]]

GRL coefficient: 1.0
Forward value: 1.2000000476837158
Observed gradient: [[-0.30000001192092896, 0.4000000059604645, -0.20000000298023224]]
Expected gradient: [[-0.30000001192092896, 0.4000000059604645, -0.20000000298023224]]

Gradient-reversal sanity check passed.


In [7]:
script_sanity_images = (
    script_batch[
        "image"
    ][:8]
    .to(
        TRAIN_DEVICE
    )
)

script_sanity_labels = (
    script_batch[
        "language_label"
    ][:8]
    .long()
    .to(
        TRAIN_DEVICE
    )
)

script_criterion = nn.CrossEntropyLoss()

target_layer4_parameter = (
    adversarial_model
    .backbone
    .layer4[-1]
    .conv2
    .weight
)


def capture_parameter_gradient(
    parameter,
):
    if parameter.grad is None:
        return torch.zeros_like(
            parameter
        ).detach().cpu()

    return (
        parameter.grad
        .detach()
        .cpu()
        .clone()
    )


def measure_script_gradients(
    grl_coefficient=None,
):
    adversarial_model.zero_grad(
        set_to_none=True
    )

    adversarial_model.train()

    embeddings = adversarial_model.encode(
        script_sanity_images
    )

    if grl_coefficient is None:
        script_logits = (
            adversarial_model
            .script_classifier(
                embeddings
            )
        )

    else:
        script_logits = (
            adversarial_model
            .classify_script(
                embeddings,
                grl_coefficient,
            )
        )

    script_loss = script_criterion(
        script_logits,
        script_sanity_labels,
    )

    script_loss.backward()

    encoder_gradient = (
        capture_parameter_gradient(
            target_layer4_parameter
        )
    )

    classifier_gradient = (
        capture_parameter_gradient(
            adversarial_model
            .script_classifier
            .weight
        )
    )

    return {
        "loss": float(
            script_loss.detach().cpu()
        ),
        "encoder_gradient": (
            encoder_gradient
        ),
        "classifier_gradient": (
            classifier_gradient
        ),
    }


direct_result = measure_script_gradients(
    grl_coefficient=None
)

zero_result = measure_script_gradients(
    grl_coefficient=0.0
)

reversed_result = measure_script_gradients(
    grl_coefficient=1.0
)

direct_encoder_gradient = (
    direct_result[
        "encoder_gradient"
    ]
)

zero_encoder_gradient = (
    zero_result[
        "encoder_gradient"
    ]
)

reversed_encoder_gradient = (
    reversed_result[
        "encoder_gradient"
    ]
)

direct_classifier_gradient = (
    direct_result[
        "classifier_gradient"
    ]
)

zero_classifier_gradient = (
    zero_result[
        "classifier_gradient"
    ]
)

reversed_classifier_gradient = (
    reversed_result[
        "classifier_gradient"
    ]
)

encoder_gradient_cosine = (
    F.cosine_similarity(
        direct_encoder_gradient.flatten(),
        reversed_encoder_gradient.flatten(),
        dim=0,
    )
)

print(
    "Script sanity labels:",
    script_sanity_labels
    .detach()
    .cpu()
    .tolist(),
)

print(
    "Direct script loss:",
    direct_result[
        "loss"
    ],
)

print(
    "Lambda-0 script loss:",
    zero_result[
        "loss"
    ],
)

print(
    "Lambda-1 script loss:",
    reversed_result[
        "loss"
    ],
)

print(
    "Direct encoder grad norm:",
    direct_encoder_gradient.norm().item(),
)

print(
    "Lambda-0 encoder grad norm:",
    zero_encoder_gradient.norm().item(),
)

print(
    "Lambda-1 encoder grad norm:",
    reversed_encoder_gradient.norm().item(),
)

print(
    "Direct vs reversed cosine:",
    encoder_gradient_cosine.item(),
)

print(
    "Max |reversed + direct|:",
    (
        reversed_encoder_gradient
        + direct_encoder_gradient
    )
    .abs()
    .max()
    .item(),
)

print(
    "Classifier grad norms:",
    {
        "direct": (
            direct_classifier_gradient
            .norm()
            .item()
        ),
        "lambda_0": (
            zero_classifier_gradient
            .norm()
            .item()
        ),
        "lambda_1": (
            reversed_classifier_gradient
            .norm()
            .item()
        ),
    },
)

assert np.isclose(
    direct_result[
        "loss"
    ],
    zero_result[
        "loss"
    ],
    atol=1e-7,
)

assert np.isclose(
    direct_result[
        "loss"
    ],
    reversed_result[
        "loss"
    ],
    atol=1e-7,
)

assert torch.allclose(
    zero_encoder_gradient,
    torch.zeros_like(
        zero_encoder_gradient
    ),
    atol=1e-8,
)

assert torch.allclose(
    reversed_encoder_gradient,
    -direct_encoder_gradient,
    atol=1e-6,
    rtol=1e-5,
)

assert torch.allclose(
    zero_classifier_gradient,
    direct_classifier_gradient,
    atol=1e-6,
    rtol=1e-5,
)

assert torch.allclose(
    reversed_classifier_gradient,
    direct_classifier_gradient,
    atol=1e-6,
    rtol=1e-5,
)

assert (
    direct_encoder_gradient.norm()
    .item()
    > 0
)

assert (
    direct_classifier_gradient.norm()
    .item()
    > 0
)

print(
    "Real-model GRL gradient check passed."
)

adversarial_model.zero_grad(
    set_to_none=True
)

Script sanity labels: [1, 1, 1, 1, 1, 1, 0, 1]
Direct script loss: 0.6849896907806396
Lambda-0 script loss: 0.6849896907806396
Lambda-1 script loss: 0.6849896907806396
Direct encoder grad norm: 0.333812415599823
Lambda-0 encoder grad norm: 0.0
Lambda-1 encoder grad norm: 0.333812415599823
Direct vs reversed cosine: -1.0003013610839844
Max |reversed + direct|: 0.0
Classifier grad norms: {'direct': 0.4681997001171112, 'lambda_0': 0.4681997001171112, 'lambda_1': 0.4681997001171112}
Real-model GRL gradient check passed.


In [8]:
metric_criterion = nn.CosineEmbeddingLoss(
    margin=0.5
)

zero_control_script_criterion = (
    nn.CrossEntropyLoss()
)

adversarial_model.zero_grad(
    set_to_none=True
)

metric_only_model = copy.deepcopy(
    adversarial_model
).to(
    TRAIN_DEVICE
)

zero_control_model = copy.deepcopy(
    adversarial_model
).to(
    TRAIN_DEVICE
)

metric_only_model.train()
zero_control_model.train()

metric_images_a = (
    metric_batch[
        "image_a"
    ]
    .to(
        TRAIN_DEVICE
    )
)

metric_images_b = (
    metric_batch[
        "image_b"
    ]
    .to(
        TRAIN_DEVICE
    )
)

metric_targets = (
    metric_batch[
        "pair_label"
    ]
    .to(
        TRAIN_DEVICE
    )
    .float()
    .mul(
        2.0
    )
    .sub(
        1.0
    )
)

control_script_images = (
    script_batch[
        "image"
    ]
    .to(
        TRAIN_DEVICE
    )
)

control_script_labels = (
    script_batch[
        "language_label"
    ]
    .long()
    .to(
        TRAIN_DEVICE
    )
)


metric_only_model.zero_grad(
    set_to_none=True
)

metric_only_embedding_a = (
    metric_only_model.encode(
        metric_images_a
    )
)

metric_only_embedding_b = (
    metric_only_model.encode(
        metric_images_b
    )
)

metric_only_loss = metric_criterion(
    metric_only_embedding_a,
    metric_only_embedding_b,
    metric_targets,
)

metric_only_loss.backward()


zero_control_model.zero_grad(
    set_to_none=True
)

zero_embedding_a = (
    zero_control_model.encode(
        metric_images_a
    )
)

zero_embedding_b = (
    zero_control_model.encode(
        metric_images_b
    )
)

zero_metric_loss = metric_criterion(
    zero_embedding_a,
    zero_embedding_b,
    metric_targets,
)

zero_script_embeddings = (
    zero_control_model.encode(
        control_script_images
    )
)

zero_script_logits = (
    zero_control_model
    .classify_script(
        zero_script_embeddings,
        grl_coefficient=0.0,
    )
)

zero_script_loss = (
    zero_control_script_criterion(
        zero_script_logits,
        control_script_labels,
    )
)

zero_total_loss = (
    zero_metric_loss
    + zero_script_loss
)

zero_total_loss.backward()


maximum_layer4_gradient_difference = (
    0.0
)

metric_layer4_gradient_norm_sq = (
    0.0
)

zero_layer4_gradient_norm_sq = (
    0.0
)

for (
    metric_parameter,
    zero_parameter,
) in zip(
    metric_only_model
    .backbone
    .layer4
    .parameters(),
    zero_control_model
    .backbone
    .layer4
    .parameters(),
):
    if not (
        metric_parameter.requires_grad
        and zero_parameter.requires_grad
    ):
        continue

    assert metric_parameter.grad is not None
    assert zero_parameter.grad is not None

    gradient_difference = (
        metric_parameter.grad
        - zero_parameter.grad
    )

    maximum_layer4_gradient_difference = max(
        maximum_layer4_gradient_difference,
        float(
            gradient_difference
            .abs()
            .max()
            .detach()
            .cpu()
        ),
    )

    metric_layer4_gradient_norm_sq += float(
        (
            metric_parameter.grad
            .detach()
            .pow(
                2
            )
            .sum()
            .cpu()
        )
    )

    zero_layer4_gradient_norm_sq += float(
        (
            zero_parameter.grad
            .detach()
            .pow(
                2
            )
            .sum()
            .cpu()
        )
    )

metric_layer4_gradient_norm = (
    metric_layer4_gradient_norm_sq
    ** 0.5
)

zero_layer4_gradient_norm = (
    zero_layer4_gradient_norm_sq
    ** 0.5
)

zero_script_head_gradient_norm = (
    zero_control_model
    .script_classifier
    .weight
    .grad
    .detach()
    .norm()
    .cpu()
    .item()
)

print(
    "Metric-only loss:",
    float(
        metric_only_loss
        .detach()
        .cpu()
    ),
)

print(
    "Zero-control metric loss:",
    float(
        zero_metric_loss
        .detach()
        .cpu()
    ),
)

print(
    "Zero-control script loss:",
    float(
        zero_script_loss
        .detach()
        .cpu()
    ),
)

print(
    "Metric-only layer4 grad norm:",
    metric_layer4_gradient_norm,
)

print(
    "Zero-control layer4 grad norm:",
    zero_layer4_gradient_norm,
)

print(
    "Maximum layer4 gradient difference:",
    maximum_layer4_gradient_difference,
)

print(
    "Zero-control script-head grad norm:",
    zero_script_head_gradient_norm,
)

assert torch.allclose(
    metric_only_loss.detach(),
    zero_metric_loss.detach(),
    atol=1e-7,
)

assert np.isclose(
    metric_layer4_gradient_norm,
    zero_layer4_gradient_norm,
    rtol=1e-5,
    atol=1e-8,
)

assert (
    maximum_layer4_gradient_difference
    < 1e-6
)

assert (
    zero_script_head_gradient_norm
    > 0
)

print(
    "Lambda-0 metric-gradient control passed."
)

del metric_only_model
del zero_control_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Metric-only loss: 0.15070068836212158
Zero-control metric loss: 0.15070068836212158
Zero-control script loss: 0.6926006078720093
Metric-only layer4 grad norm: 1.9528936898641551
Zero-control layer4 grad norm: 1.9528936898641551
Maximum layer4 gradient difference: 0.0
Zero-control script-head grad norm: 0.13706953823566437
Lambda-0 metric-gradient control passed.


In [9]:
ENCODER_LR = 1e-5
SCRIPT_HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_MAX_NORM = 5.0

ZERO_ADVERSARIAL_COEFFICIENT = 0.0


validation_dataset = QUWIDataset(
    metadata=validation_df,
    image_dir=IMAGE_DIR,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)


def build_fresh_adversarial_model():
    torch.manual_seed(
        SEED
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            SEED
        )

    model = (
        ScriptAdversarialWriterModel()
        .to(
            TRAIN_DEVICE
        )
    )

    load_result = (
        model
        .backbone
        .load_state_dict(
            ssl_backbone_state_dict,
            strict=True,
        )
    )

    assert load_result.missing_keys == []
    assert load_result.unexpected_keys == []

    return model


def extract_embedding_map(
    model,
    loader,
):
    model.eval()

    embedding_map = {}

    with torch.no_grad():
        for batch in loader:
            images = (
                batch[
                    "image"
                ]
                .to(
                    TRAIN_DEVICE
                )
            )

            embeddings = model.encode(
                images
            )

            filenames = batch[
                "filename"
            ]

            for filename, embedding in zip(
                filenames,
                embeddings,
            ):
                embedding_map[
                    filename
                ] = (
                    embedding
                    .detach()
                    .cpu()
                    .numpy()
                )

    return embedding_map


def calculate_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = 1.0 - tpr

    eer_index = np.nanargmin(
        np.abs(
            fpr
            - fnr
        )
    )

    eer = (
        fpr[
            eer_index
        ]
        + fnr[
            eer_index
        ]
    ) / 2.0

    threshold = thresholds[
        eer_index
    ]

    return (
        float(
            eer
        ),
        float(
            threshold
        ),
    )


def evaluate_validation_verification(
    model,
):
    embedding_map = extract_embedding_map(
        model,
        validation_loader,
    )

    embeddings_a = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_a"
                ]
            )
        ]
    )

    embeddings_b = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_b"
                ]
            )
        ]
    )

    scores = np.sum(
        embeddings_a
        * embeddings_b,
        axis=1,
    )

    labels = (
        validation_pairs_df[
            "pair_label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, eer_threshold = calculate_eer(
        labels,
        scores,
    )

    return {
        "auc": float(
            auc
        ),
        "eer": float(
            eer
        ),
        "eer_threshold": float(
            eer_threshold
        ),
    }


def train_joint_epoch(
    model,
    epoch,
    grl_coefficient,
    encoder_optimizer,
    script_optimizer,
):
    (
        epoch_pairs_df,
        metric_loader,
    ) = create_metric_loader(
        epoch
    )

    script_loader = create_script_loader(
        epoch
    )

    assert len(
        metric_loader
    ) == len(
        script_loader
    )

    assert len(
        epoch_pairs_df
    ) == 452

    model.train()

    metric_loss_sum = 0.0
    script_loss_sum = 0.0

    script_correct = 0
    script_total = 0

    encoder_grad_norm_sum = 0.0

    encoder_parameters = [
        parameter
        for parameter in (
            model
            .backbone
            .layer4
            .parameters()
        )
        if parameter.requires_grad
    ]

    script_parameters = [
        parameter
        for parameter in (
            model
            .script_classifier
            .parameters()
        )
        if parameter.requires_grad
    ]

    for (
        metric_batch_current,
        script_batch_current,
    ) in zip(
        metric_loader,
        script_loader,
    ):
        encoder_optimizer.zero_grad(
            set_to_none=True
        )

        script_optimizer.zero_grad(
            set_to_none=True
        )

        image_a = (
            metric_batch_current[
                "image_a"
            ]
            .to(
                TRAIN_DEVICE
            )
        )

        image_b = (
            metric_batch_current[
                "image_b"
            ]
            .to(
                TRAIN_DEVICE
            )
        )

        metric_targets = (
            metric_batch_current[
                "pair_label"
            ]
            .to(
                TRAIN_DEVICE
            )
            .float()
            .mul(
                2.0
            )
            .sub(
                1.0
            )
        )

        embedding_a = model.encode(
            image_a
        )

        embedding_b = model.encode(
            image_b
        )

        metric_loss = metric_criterion(
            embedding_a,
            embedding_b,
            metric_targets,
        )

        script_images = (
            script_batch_current[
                "image"
            ]
            .to(
                TRAIN_DEVICE
            )
        )

        script_labels = (
            script_batch_current[
                "language_label"
            ]
            .long()
            .to(
                TRAIN_DEVICE
            )
        )

        script_embeddings = model.encode(
            script_images
        )

        script_logits = (
            model.classify_script(
                script_embeddings,
                grl_coefficient,
            )
        )

        script_loss = script_criterion(
            script_logits,
            script_labels,
        )

        total_loss = (
            metric_loss
            + script_loss
        )

        total_loss.backward()

        encoder_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                encoder_parameters,
                max_norm=(
                    GRAD_CLIP_MAX_NORM
                ),
            )
        )

        torch.nn.utils.clip_grad_norm_(
            script_parameters,
            max_norm=(
                GRAD_CLIP_MAX_NORM
            ),
        )

        encoder_optimizer.step()
        script_optimizer.step()

        metric_loss_sum += float(
            metric_loss
            .detach()
            .cpu()
        )

        script_loss_sum += float(
            script_loss
            .detach()
            .cpu()
        )

        script_predictions = (
            script_logits
            .argmax(
                dim=1
            )
        )

        script_correct += int(
            (
                script_predictions
                == script_labels
            )
            .sum()
            .detach()
            .cpu()
        )

        script_total += int(
            script_labels.shape[
                0
            ]
        )

        encoder_grad_norm_sum += float(
            encoder_grad_norm
            .detach()
            .cpu()
        )

    batch_count = len(
        metric_loader
    )

    return {
        "metric_loss": (
            metric_loss_sum
            / batch_count
        ),
        "script_loss": (
            script_loss_sum
            / batch_count
        ),
        "script_accuracy": (
            script_correct
            / script_total
        ),
        "encoder_grad_norm": (
            encoder_grad_norm_sum
            / batch_count
        ),
    }


fresh_control_model = (
    build_fresh_adversarial_model()
)

initial_validation_metrics = (
    evaluate_validation_verification(
        fresh_control_model
    )
)

print(
    "Validation images:",
    len(
        validation_dataset
    ),
)

print(
    "Validation embeddings:",
    len(
        extract_embedding_map(
            fresh_control_model,
            validation_loader,
        )
    ),
)

print(
    "Validation pairs:",
    len(
        validation_pairs_df
    ),
)

print(
    "Initial SSL validation AUC:",
    initial_validation_metrics[
        "auc"
    ],
)

print(
    "Initial SSL validation EER:",
    initial_validation_metrics[
        "eer"
    ],
)

assert len(
    validation_dataset
) == 224

assert len(
    validation_pairs_df
) == 18_816

assert (
    0.0
    <= initial_validation_metrics[
        "auc"
    ]
    <= 1.0
)

assert (
    0.0
    <= initial_validation_metrics[
        "eer"
    ]
    <= 1.0
)

del fresh_control_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Validation images: 224
Validation embeddings: 224
Validation pairs: 18816
Initial SSL validation AUC: 0.6737687783446711
Initial SSL validation EER: 0.38871753246753243


In [10]:
ZERO_CONTROL_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "resnet18_script_adversarial_lambda0_best.pt"
)

zero_control_model = (
    build_fresh_adversarial_model()
)

encoder_optimizer = torch.optim.AdamW(
    zero_control_model
    .backbone
    .layer4
    .parameters(),
    lr=ENCODER_LR,
    weight_decay=WEIGHT_DECAY,
)

script_optimizer = torch.optim.AdamW(
    zero_control_model
    .script_classifier
    .parameters(),
    lr=SCRIPT_HEAD_LR,
    weight_decay=WEIGHT_DECAY,
)

zero_control_history = []

best_zero_control_auc = (
    -np.inf
)

best_zero_control_eer = (
    np.inf
)

best_zero_control_epoch = None


for epoch in range(
    1,
    EPOCHS + 1,
):
    training_metrics = train_joint_epoch(
        model=zero_control_model,
        epoch=epoch,
        grl_coefficient=(
            ZERO_ADVERSARIAL_COEFFICIENT
        ),
        encoder_optimizer=(
            encoder_optimizer
        ),
        script_optimizer=(
            script_optimizer
        ),
    )

    validation_metrics = (
        evaluate_validation_verification(
            zero_control_model
        )
    )

    current_auc = (
        validation_metrics[
            "auc"
        ]
    )

    current_eer = (
        validation_metrics[
            "eer"
        ]
    )

    auc_improved = (
        current_auc
        > best_zero_control_auc
    )

    auc_tied = np.isclose(
        current_auc,
        best_zero_control_auc,
        rtol=0.0,
        atol=1e-12,
    )

    eer_improved_on_tie = (
        auc_tied
        and current_eer
        < best_zero_control_eer
    )

    checkpoint_selected = (
        auc_improved
        or eer_improved_on_tie
    )

    if checkpoint_selected:
        best_zero_control_auc = (
            current_auc
        )

        best_zero_control_eer = (
            current_eer
        )

        best_zero_control_epoch = (
            epoch
        )

        torch.save(
            {
                "epoch": epoch,
                "grl_coefficient": (
                    ZERO_ADVERSARIAL_COEFFICIENT
                ),
                "encoder_lr": (
                    ENCODER_LR
                ),
                "script_head_lr": (
                    SCRIPT_HEAD_LR
                ),
                "weight_decay": (
                    WEIGHT_DECAY
                ),
                "model_state_dict": (
                    zero_control_model
                    .state_dict()
                ),
                "validation_auc": (
                    current_auc
                ),
                "validation_eer": (
                    current_eer
                ),
                "validation_eer_threshold": (
                    validation_metrics[
                        "eer_threshold"
                    ]
                ),
            },
            ZERO_CONTROL_CHECKPOINT_PATH,
        )

    zero_control_history.append(
        {
            "epoch": epoch,
            "grl_coefficient": (
                ZERO_ADVERSARIAL_COEFFICIENT
            ),
            "metric_loss": (
                training_metrics[
                    "metric_loss"
                ]
            ),
            "script_loss": (
                training_metrics[
                    "script_loss"
                ]
            ),
            "script_accuracy": (
                training_metrics[
                    "script_accuracy"
                ]
            ),
            "encoder_grad_norm": (
                training_metrics[
                    "encoder_grad_norm"
                ]
            ),
            "validation_auc": (
                current_auc
            ),
            "validation_eer": (
                current_eer
            ),
            "checkpoint_selected": (
                checkpoint_selected
            ),
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Metric loss "
        f"{training_metrics['metric_loss']:.6f} | "
        f"Script loss "
        f"{training_metrics['script_loss']:.6f} | "
        f"Script acc "
        f"{training_metrics['script_accuracy']:.4f} | "
        f"Grad norm "
        f"{training_metrics['encoder_grad_norm']:.4f} | "
        f"Val AUC "
        f"{current_auc:.6f} | "
        f"Val EER "
        f"{100.0 * current_eer:.4f}%"
        + (
            " | selected"
            if checkpoint_selected
            else ""
        )
    )


zero_control_history_df = pd.DataFrame(
    zero_control_history
)

print()

print(
    "Best zero-control epoch:",
    best_zero_control_epoch,
)

print(
    "Best zero-control validation AUC:",
    best_zero_control_auc,
)

print(
    "Best zero-control validation EER:",
    best_zero_control_eer,
)

print(
    "Checkpoint:",
    ZERO_CONTROL_CHECKPOINT_PATH,
)

assert len(
    zero_control_history_df
) == EPOCHS

assert best_zero_control_epoch is not None

assert ZERO_CONTROL_CHECKPOINT_PATH.exists()

Epoch 01 | Metric loss 0.196796 | Script loss 0.653497 | Script acc 0.8009 | Grad norm 1.8937 | Val AUC 0.706767 | Val EER 35.7576% | selected
Epoch 02 | Metric loss 0.177247 | Script loss 0.572672 | Script acc 0.9480 | Grad norm 1.8492 | Val AUC 0.720991 | Val EER 34.4968% | selected
Epoch 03 | Metric loss 0.164112 | Script loss 0.510898 | Script acc 0.9303 | Grad norm 2.0074 | Val AUC 0.718100 | Val EER 35.1218%
Epoch 04 | Metric loss 0.151657 | Script loss 0.461145 | Script acc 0.9613 | Grad norm 1.9700 | Val AUC 0.734852 | Val EER 33.0330% | selected
Epoch 05 | Metric loss 0.152637 | Script loss 0.416804 | Script acc 0.9458 | Grad norm 2.0142 | Val AUC 0.729148 | Val EER 33.6282%
Epoch 06 | Metric loss 0.139087 | Script loss 0.387136 | Script acc 0.9403 | Grad norm 2.3423 | Val AUC 0.738153 | Val EER 33.3009% | selected
Epoch 07 | Metric loss 0.130787 | Script loss 0.368543 | Script acc 0.9436 | Grad norm 2.1354 | Val AUC 0.742373 | Val EER 33.4334% | selected
Epoch 08 | Metric los

In [11]:
best_zero_control_checkpoint = torch.load(
    ZERO_CONTROL_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

best_zero_control_model = (
    build_fresh_adversarial_model()
)

best_zero_control_model.load_state_dict(
    best_zero_control_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

best_zero_control_model = (
    best_zero_control_model
    .to(
        TRAIN_DEVICE
    )
)

best_zero_control_model.eval()


validation_embedding_map = (
    extract_embedding_map(
        best_zero_control_model,
        validation_loader,
    )
)

validation_embeddings_a = np.stack(
    [
        validation_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

validation_embeddings_b = np.stack(
    [
        validation_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)

validation_scores = np.sum(
    validation_embeddings_a
    * validation_embeddings_b,
    axis=1,
)

validation_labels = (
    validation_pairs_df[
        "pair_label"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


def calculate_interpolated_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = 1.0 - tpr

    difference = (
        fnr
        - fpr
    )

    exact_indices = np.where(
        np.isclose(
            difference,
            0.0,
            atol=1e-15,
        )
    )[0]

    if len(
        exact_indices
    ) > 0:
        index = int(
            exact_indices[
                0
            ]
        )

        return (
            float(
                fpr[
                    index
                ]
            ),
            float(
                thresholds[
                    index
                ]
            ),
        )

    crossing_indices = np.where(
        difference[:-1]
        * difference[1:]
        < 0
    )[0]

    assert len(
        crossing_indices
    ) > 0

    index = int(
        crossing_indices[
            0
        ]
    )

    difference_a = difference[
        index
    ]

    difference_b = difference[
        index
        + 1
    ]

    interpolation_fraction = (
        difference_a
        / (
            difference_a
            - difference_b
        )
    )

    eer = (
        fpr[
            index
        ]
        + interpolation_fraction
        * (
            fpr[
                index
                + 1
            ]
            - fpr[
                index
            ]
        )
    )

    eer_threshold = (
        thresholds[
            index
        ]
        + interpolation_fraction
        * (
            thresholds[
                index
                + 1
            ]
            - thresholds[
                index
            ]
        )
    )

    return (
        float(
            eer
        ),
        float(
            eer_threshold
        ),
    )


validation_auc = roc_auc_score(
    validation_labels,
    validation_scores,
)

nearest_eer, nearest_threshold = (
    calculate_eer(
        validation_labels,
        validation_scores,
    )
)

(
    interpolated_eer,
    interpolated_threshold,
) = calculate_interpolated_eer(
    validation_labels,
    validation_scores,
)

print(
    "Validation AUC:",
    validation_auc,
)

print(
    "Nearest-point EER:",
    nearest_eer,
)

print(
    "Nearest-point EER (%):",
    100.0
    * nearest_eer,
)

print(
    "Nearest-point threshold:",
    nearest_threshold,
)

print(
    "Interpolated EER:",
    interpolated_eer,
)

print(
    "Interpolated EER (%):",
    100.0
    * interpolated_eer,
)

print(
    "Interpolated threshold:",
    interpolated_threshold,
)

print(
    "Recorded Notebook-14 EER (%):",
    32.9978,
)

print(
    "Difference from recorded EER (percentage points):",
    (
        100.0
        * interpolated_eer
        - 32.9978
    ),
)

assert np.isclose(
    validation_auc,
    0.7441469864460937,
    atol=1e-12,
)

Validation AUC: 0.7441469864460937
Nearest-point EER: 0.3301677489177489
Nearest-point EER (%): 33.016774891774894
Nearest-point threshold: 0.6185126304626465
Interpolated EER: 0.329978354978355
Interpolated EER (%): 32.997835497835496
Interpolated threshold: 0.6185125697742809
Recorded Notebook-14 EER (%): 32.9978
Difference from recorded EER (percentage points): 3.549783549772201e-05


In [12]:
hybrid_checkpoint = torch.load(
    HYBRID_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

zero_control_checkpoint = torch.load(
    ZERO_CONTROL_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)


def get_model_state_dict(
    checkpoint,
):
    if "model_state_dict" in checkpoint:
        return checkpoint[
            "model_state_dict"
        ]

    if "state_dict" in checkpoint:
        return checkpoint[
            "state_dict"
        ]

    raise KeyError(
        "No model state dictionary found."
    )


def extract_layer4_state(
    state_dict,
):
    layer4_state = {}

    for key, value in (
        state_dict.items()
    ):
        marker = "layer4."

        marker_index = key.find(
            marker
        )

        if marker_index == -1:
            continue

        normalized_key = key[
            marker_index:
        ]

        layer4_state[
            normalized_key
        ] = value.detach().cpu()

    return layer4_state


hybrid_state_dict = get_model_state_dict(
    hybrid_checkpoint
)

zero_control_state_dict = (
    get_model_state_dict(
        zero_control_checkpoint
    )
)

hybrid_layer4_state = (
    extract_layer4_state(
        hybrid_state_dict
    )
)

zero_control_layer4_state = (
    extract_layer4_state(
        zero_control_state_dict
    )
)

print(
    "Hybrid checkpoint keys:",
    list(
        hybrid_checkpoint.keys()
    ),
)

print(
    "Zero-control checkpoint keys:",
    list(
        zero_control_checkpoint.keys()
    ),
)

print(
    "Hybrid checkpoint epoch:",
    hybrid_checkpoint.get(
        "epoch"
    ),
)

print(
    "Zero-control checkpoint epoch:",
    zero_control_checkpoint.get(
        "epoch"
    ),
)

print(
    "Hybrid layer4 tensors:",
    len(
        hybrid_layer4_state
    ),
)

print(
    "Zero-control layer4 tensors:",
    len(
        zero_control_layer4_state
    ),
)

assert set(
    hybrid_layer4_state
) == set(
    zero_control_layer4_state
)

maximum_absolute_difference = 0.0

different_tensor_count = 0

for key in sorted(
    hybrid_layer4_state
):
    hybrid_tensor = (
        hybrid_layer4_state[
            key
        ]
    )

    zero_tensor = (
        zero_control_layer4_state[
            key
        ]
    )

    tensor_difference = (
        hybrid_tensor
        - zero_tensor
    ).abs()

    maximum_tensor_difference = float(
        tensor_difference
        .max()
        .item()
    )

    maximum_absolute_difference = max(
        maximum_absolute_difference,
        maximum_tensor_difference,
    )

    if not torch.equal(
        hybrid_tensor,
        zero_tensor,
    ):
        different_tensor_count += 1


print(
    "Different layer4 tensors:",
    different_tensor_count,
)

print(
    "Maximum absolute layer4 difference:",
    maximum_absolute_difference,
)

if (
    different_tensor_count
    == 0
):
    print(
        "Layer4 is bitwise identical "
        "to the Notebook-14 hybrid checkpoint."
    )

else:
    print(
        "Layer4 is not bitwise identical; "
        "numerical difference requires inspection."
    )

Hybrid checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'training_metrics', 'validation_metrics', 'configuration']
Zero-control checkpoint keys: ['epoch', 'grl_coefficient', 'encoder_lr', 'script_head_lr', 'weight_decay', 'model_state_dict', 'validation_auc', 'validation_eer', 'validation_eer_threshold']
Hybrid checkpoint epoch: 8
Zero-control checkpoint epoch: 8
Hybrid layer4 tensors: 30
Zero-control layer4 tensors: 30
Different layer4 tensors: 0
Maximum absolute layer4 difference: 0.0
Layer4 is bitwise identical to the Notebook-14 hybrid checkpoint.


In [13]:
def calculate_canonical_eer(
    labels,
    scores,
):
    return calculate_interpolated_eer(
        labels,
        scores,
    )


def evaluate_validation_verification(
    model,
):
    embedding_map = extract_embedding_map(
        model,
        validation_loader,
    )

    embeddings_a = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_a"
                ]
            )
        ]
    )

    embeddings_b = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_b"
                ]
            )
        ]
    )

    scores = np.sum(
        embeddings_a
        * embeddings_b,
        axis=1,
    )

    labels = (
        validation_pairs_df[
            "pair_label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, eer_threshold = (
        calculate_canonical_eer(
            labels,
            scores,
        )
    )

    return {
        "auc": float(
            auc
        ),
        "eer": float(
            eer
        ),
        "eer_threshold": float(
            eer_threshold
        ),
    }


canonical_zero_metrics = (
    evaluate_validation_verification(
        best_zero_control_model
    )
)

print(
    "Canonical zero-control AUC:",
    canonical_zero_metrics[
        "auc"
    ],
)

print(
    "Canonical zero-control EER:",
    canonical_zero_metrics[
        "eer"
    ],
)

print(
    "Canonical zero-control EER (%):",
    100.0
    * canonical_zero_metrics[
        "eer"
    ],
)

assert np.isclose(
    canonical_zero_metrics[
        "auc"
    ],
    0.7441469864460937,
    atol=1e-12,
)

assert np.isclose(
    100.0
    * canonical_zero_metrics[
        "eer"
    ],
    32.997835497835496,
    atol=1e-10,
)

Canonical zero-control AUC: 0.7441469864460937
Canonical zero-control EER: 0.329978354978355
Canonical zero-control EER (%): 32.997835497835496


In [14]:
GRADIENT_SCALE_BATCHES = 12


def layer4_gradient_norm(
    model,
):
    total_squared_norm = 0.0

    for parameter in (
        model
        .backbone
        .layer4
        .parameters()
    ):
        if (
            parameter.requires_grad
            and parameter.grad
            is not None
        ):
            total_squared_norm += float(
                parameter.grad
                .detach()
                .pow(
                    2
                )
                .sum()
                .cpu()
            )

    return (
        total_squared_norm
        ** 0.5
    )


gradient_scale_model = (
    build_fresh_adversarial_model()
)

gradient_scale_model.train()

(
    _,
    gradient_metric_loader,
) = create_metric_loader(
    1
)

gradient_script_loader = (
    create_script_loader(
        1
    )
)

gradient_scale_rows = []


for batch_index, (
    metric_batch_current,
    script_batch_current,
) in enumerate(
    zip(
        gradient_metric_loader,
        gradient_script_loader,
    ),
    start=1,
):
    if (
        batch_index
        > GRADIENT_SCALE_BATCHES
    ):
        break

    metric_images_a = (
        metric_batch_current[
            "image_a"
        ]
        .to(
            TRAIN_DEVICE
        )
    )

    metric_images_b = (
        metric_batch_current[
            "image_b"
        ]
        .to(
            TRAIN_DEVICE
        )
    )

    metric_targets = (
        metric_batch_current[
            "pair_label"
        ]
        .to(
            TRAIN_DEVICE
        )
        .float()
        .mul(
            2.0
        )
        .sub(
            1.0
        )
    )

    gradient_scale_model.zero_grad(
        set_to_none=True
    )

    metric_embedding_a = (
        gradient_scale_model.encode(
            metric_images_a
        )
    )

    metric_embedding_b = (
        gradient_scale_model.encode(
            metric_images_b
        )
    )

    metric_loss = metric_criterion(
        metric_embedding_a,
        metric_embedding_b,
        metric_targets,
    )

    metric_loss.backward()

    metric_grad_norm = (
        layer4_gradient_norm(
            gradient_scale_model
        )
    )


    script_images = (
        script_batch_current[
            "image"
        ]
        .to(
            TRAIN_DEVICE
        )
    )

    script_labels = (
        script_batch_current[
            "language_label"
        ]
        .long()
        .to(
            TRAIN_DEVICE
        )
    )

    gradient_scale_model.zero_grad(
        set_to_none=True
    )

    script_embeddings = (
        gradient_scale_model.encode(
            script_images
        )
    )

    script_logits = (
        gradient_scale_model
        .script_classifier(
            script_embeddings
        )
    )

    script_loss = script_criterion(
        script_logits,
        script_labels,
    )

    script_loss.backward()

    script_grad_norm = (
        layer4_gradient_norm(
            gradient_scale_model
        )
    )

    equal_gradient_lambda = (
        metric_grad_norm
        / script_grad_norm
        if script_grad_norm > 0.0
        else np.nan
    )

    gradient_scale_rows.append(
        {
            "batch": batch_index,
            "metric_loss": float(
                metric_loss
                .detach()
                .cpu()
            ),
            "script_loss": float(
                script_loss
                .detach()
                .cpu()
            ),
            "metric_grad_norm": (
                metric_grad_norm
            ),
            "script_grad_norm_lambda1": (
                script_grad_norm
            ),
            "script_to_metric_ratio": (
                script_grad_norm
                / metric_grad_norm
            ),
            "equal_gradient_lambda": (
                equal_gradient_lambda
            ),
        }
    )


gradient_scale_df = pd.DataFrame(
    gradient_scale_rows
)

median_metric_grad = float(
    gradient_scale_df[
        "metric_grad_norm"
    ]
    .median()
)

median_script_grad = float(
    gradient_scale_df[
        "script_grad_norm_lambda1"
    ]
    .median()
)

median_equal_lambda = float(
    gradient_scale_df[
        "equal_gradient_lambda"
    ]
    .median()
)

lambda_for_10_percent = (
    0.10
    * median_equal_lambda
)

lambda_for_25_percent = (
    0.25
    * median_equal_lambda
)

lambda_for_50_percent = (
    0.50
    * median_equal_lambda
)


print(
    gradient_scale_df[
        [
            "batch",
            "metric_grad_norm",
            "script_grad_norm_lambda1",
            "script_to_metric_ratio",
            "equal_gradient_lambda",
        ]
    ].round(
        6
    )
    .to_string(
        index=False
    )
)

print()

print(
    "Median metric grad norm:",
    median_metric_grad,
)

print(
    "Median script grad norm at lambda=1:",
    median_script_grad,
)

print(
    "Median equal-gradient lambda:",
    median_equal_lambda,
)

print(
    "Lambda for ~10% script/metric gradient:",
    lambda_for_10_percent,
)

print(
    "Lambda for ~25% script/metric gradient:",
    lambda_for_25_percent,
)

print(
    "Lambda for ~50% script/metric gradient:",
    lambda_for_50_percent,
)

assert len(
    gradient_scale_df
) == GRADIENT_SCALE_BATCHES

assert (
    gradient_scale_df[
        "metric_grad_norm"
    ]
    > 0
).all()

assert (
    gradient_scale_df[
        "script_grad_norm_lambda1"
    ]
    > 0
).all()


del gradient_scale_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

 batch  metric_grad_norm  script_grad_norm_lambda1  script_to_metric_ratio  equal_gradient_lambda
     1          1.952894                  0.251544                0.128806               7.763627
     2          1.444645                  0.293335                0.203050               4.924897
     3          1.548368                  0.388529                0.250928               3.985211
     4          1.767966                  0.431675                0.244165               4.095593
     5          2.394978                  0.318676                0.133060               7.515411
     6          1.518215                  0.594648                0.391676               2.553133
     7          1.677949                  0.332624                0.198233               5.044575
     8          1.924768                  0.394390                0.204903               4.880365
     9          2.814695                  0.285906                0.101576               9.844840
    10          2.57

In [15]:
def run_adversarial_experiment(
    grl_coefficient,
    checkpoint_name,
):
    model = build_fresh_adversarial_model()

    encoder_optimizer = torch.optim.AdamW(
        model
        .backbone
        .layer4
        .parameters(),
        lr=ENCODER_LR,
        weight_decay=WEIGHT_DECAY,
    )

    script_optimizer = torch.optim.AdamW(
        model
        .script_classifier
        .parameters(),
        lr=SCRIPT_HEAD_LR,
        weight_decay=WEIGHT_DECAY,
    )

    checkpoint_path = (
        CHECKPOINT_DIR
        / checkpoint_name
    )

    history = []

    best_auc = -np.inf
    best_eer = np.inf
    best_epoch = None

    for epoch in range(
        1,
        EPOCHS + 1,
    ):
        training_metrics = train_joint_epoch(
            model=model,
            epoch=epoch,
            grl_coefficient=grl_coefficient,
            encoder_optimizer=encoder_optimizer,
            script_optimizer=script_optimizer,
        )

        validation_metrics = (
            evaluate_validation_verification(
                model
            )
        )

        current_auc = (
            validation_metrics[
                "auc"
            ]
        )

        current_eer = (
            validation_metrics[
                "eer"
            ]
        )

        auc_improved = (
            current_auc
            > best_auc
        )

        auc_tied = np.isclose(
            current_auc,
            best_auc,
            rtol=0.0,
            atol=1e-12,
        )

        eer_improved_on_tie = (
            auc_tied
            and current_eer
            < best_eer
        )

        checkpoint_selected = (
            auc_improved
            or eer_improved_on_tie
        )

        if checkpoint_selected:
            best_auc = current_auc
            best_eer = current_eer
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "grl_coefficient": (
                        grl_coefficient
                    ),
                    "encoder_lr": (
                        ENCODER_LR
                    ),
                    "script_head_lr": (
                        SCRIPT_HEAD_LR
                    ),
                    "weight_decay": (
                        WEIGHT_DECAY
                    ),
                    "model_state_dict": (
                        model
                        .state_dict()
                    ),
                    "validation_auc": (
                        current_auc
                    ),
                    "validation_eer": (
                        current_eer
                    ),
                    "validation_eer_threshold": (
                        validation_metrics[
                            "eer_threshold"
                        ]
                    ),
                },
                checkpoint_path,
            )

        history.append(
            {
                "epoch": epoch,
                "grl_coefficient": (
                    grl_coefficient
                ),
                "metric_loss": (
                    training_metrics[
                        "metric_loss"
                    ]
                ),
                "script_loss": (
                    training_metrics[
                        "script_loss"
                    ]
                ),
                "script_accuracy": (
                    training_metrics[
                        "script_accuracy"
                    ]
                ),
                "encoder_grad_norm": (
                    training_metrics[
                        "encoder_grad_norm"
                    ]
                ),
                "validation_auc": (
                    current_auc
                ),
                "validation_eer": (
                    current_eer
                ),
                "checkpoint_selected": (
                    checkpoint_selected
                ),
            }
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Lambda {grl_coefficient:.3f} | "
            f"Metric loss "
            f"{training_metrics['metric_loss']:.6f} | "
            f"Script loss "
            f"{training_metrics['script_loss']:.6f} | "
            f"Script acc "
            f"{training_metrics['script_accuracy']:.4f} | "
            f"Grad norm "
            f"{training_metrics['encoder_grad_norm']:.4f} | "
            f"Val AUC "
            f"{current_auc:.6f} | "
            f"Val EER "
            f"{100.0 * current_eer:.4f}%"
            + (
                " | selected"
                if checkpoint_selected
                else ""
            )
        )

    history_df = pd.DataFrame(
        history
    )

    print()

    print(
        "Best epoch:",
        best_epoch,
    )

    print(
        "Best validation AUC:",
        best_auc,
    )

    print(
        "Best validation EER (%):",
        100.0
        * best_eer,
    )

    print(
        "Checkpoint:",
        checkpoint_path,
    )

    assert len(
        history_df
    ) == EPOCHS

    assert best_epoch is not None

    assert checkpoint_path.exists()

    return {
        "model": model,
        "history": history_df,
        "best_epoch": best_epoch,
        "best_auc": best_auc,
        "best_eer": best_eer,
        "checkpoint_path": (
            checkpoint_path
        ),
    }


print(
    "Adversarial experiment runner ready."
)

Adversarial experiment runner ready.


In [16]:
FIRST_ADVERSARIAL_COEFFICIENT = 0.5

lambda05_result = run_adversarial_experiment(
    grl_coefficient=(
        FIRST_ADVERSARIAL_COEFFICIENT
    ),
    checkpoint_name=(
        "resnet18_script_adversarial_lambda0p5_best.pt"
    ),
)


lambda05_auc_change = (
    lambda05_result[
        "best_auc"
    ]
    - canonical_zero_metrics[
        "auc"
    ]
)

lambda05_eer_change_pp = (
    100.0
    * (
        lambda05_result[
            "best_eer"
        ]
        - canonical_zero_metrics[
            "eer"
        ]
    )
)


print()

print(
    "Zero-control AUC:",
    canonical_zero_metrics[
        "auc"
    ],
)

print(
    "Lambda-0.5 best AUC:",
    lambda05_result[
        "best_auc"
    ],
)

print(
    "AUC change vs zero-control:",
    lambda05_auc_change,
)

print()

print(
    "Zero-control EER (%):",
    100.0
    * canonical_zero_metrics[
        "eer"
    ],
)

print(
    "Lambda-0.5 best EER (%):",
    100.0
    * lambda05_result[
        "best_eer"
    ],
)

print(
    "EER change vs zero-control "
    "(percentage points):",
    lambda05_eer_change_pp,
)

Epoch 01 | Lambda 0.500 | Metric loss 0.197640 | Script loss 0.664384 | Script acc 0.7622 | Grad norm 1.9060 | Val AUC 0.713637 | Val EER 35.4167% | selected
Epoch 02 | Lambda 0.500 | Metric loss 0.181993 | Script loss 0.643169 | Script acc 0.7998 | Grad norm 1.8443 | Val AUC 0.738843 | Val EER 33.9286% | selected
Epoch 03 | Lambda 0.500 | Metric loss 0.168522 | Script loss 0.665236 | Script acc 0.6527 | Grad norm 2.2259 | Val AUC 0.744284 | Val EER 33.0357% | selected
Epoch 04 | Lambda 0.500 | Metric loss 0.157855 | Script loss 0.696265 | Script acc 0.4889 | Grad norm 2.1131 | Val AUC 0.757581 | Val EER 30.9524% | selected
Epoch 05 | Lambda 0.500 | Metric loss 0.158539 | Script loss 0.711145 | Script acc 0.4004 | Grad norm 2.0703 | Val AUC 0.767625 | Val EER 30.2760% | selected
Epoch 06 | Lambda 0.500 | Metric loss 0.145022 | Script loss 0.700242 | Script acc 0.4768 | Grad norm 2.3422 | Val AUC 0.769506 | Val EER 29.8377% | selected
Epoch 07 | Lambda 0.500 | Metric loss 0.135387 | Scr

In [17]:
lambda05_checkpoint = torch.load(
    lambda05_result[
        "checkpoint_path"
    ],
    map_location="cpu",
    weights_only=False,
)

lambda05_best_model = (
    build_fresh_adversarial_model()
)

lambda05_best_model.load_state_dict(
    lambda05_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

lambda05_best_model = (
    lambda05_best_model
    .to(
        TRAIN_DEVICE
    )
)

lambda05_best_model.eval()


development_probe_loader = DataLoader(
    script_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)


def extract_probe_embeddings(
    model,
    loader,
):
    model.eval()

    embedding_rows = []

    with torch.no_grad():
        for batch in loader:
            images = (
                batch[
                    "image"
                ]
                .to(
                    TRAIN_DEVICE
                )
            )

            embeddings = model.encode(
                images
            )

            embeddings = (
                embeddings
                .detach()
                .cpu()
                .numpy()
            )

            for index in range(
                len(
                    embeddings
                )
            ):
                embedding_rows.append(
                    {
                        "filename": (
                            batch[
                                "filename"
                            ][
                                index
                            ]
                        ),
                        "writer": int(
                            batch[
                                "writer_id"
                            ][
                                index
                            ]
                        ),
                        "language_label": int(
                            batch[
                                "language_label"
                            ][
                                index
                            ]
                        ),
                        "language": (
                            batch[
                                "language"
                            ][
                                index
                            ]
                        ),
                        "embedding": (
                            embeddings[
                                index
                            ]
                        ),
                    }
                )

    return embedding_rows


lambda05_development_rows = (
    extract_probe_embeddings(
        lambda05_best_model,
        development_probe_loader,
    )
)

lambda05_validation_rows = (
    extract_probe_embeddings(
        lambda05_best_model,
        validation_loader,
    )
)


lambda05_development_embeddings = (
    np.stack(
        [
            row[
                "embedding"
            ]
            for row in (
                lambda05_development_rows
            )
        ]
    )
)

lambda05_validation_embeddings = (
    np.stack(
        [
            row[
                "embedding"
            ]
            for row in (
                lambda05_validation_rows
            )
        ]
    )
)

lambda05_development_script_labels = (
    np.array(
        [
            row[
                "language_label"
            ]
            for row in (
                lambda05_development_rows
            )
        ],
        dtype=np.int64,
    )
)

lambda05_validation_script_labels = (
    np.array(
        [
            row[
                "language_label"
            ]
            for row in (
                lambda05_validation_rows
            )
        ],
        dtype=np.int64,
    )
)


print(
    "Best lambda-0.5 epoch:",
    lambda05_checkpoint[
        "epoch"
    ],
)

print(
    "Development embedding shape:",
    lambda05_development_embeddings.shape,
)

print(
    "Validation embedding shape:",
    lambda05_validation_embeddings.shape,
)

print(
    "Development script counts:",
    np.bincount(
        lambda05_development_script_labels
    ).tolist(),
)

print(
    "Validation script counts:",
    np.bincount(
        lambda05_validation_script_labels
    ).tolist(),
)

print(
    "Development writers:",
    len(
        set(
            row[
                "writer"
            ]
            for row in (
                lambda05_development_rows
            )
        )
    ),
)

print(
    "Validation writers:",
    len(
        set(
            row[
                "writer"
            ]
            for row in (
                lambda05_validation_rows
            )
        )
    ),
)


assert lambda05_checkpoint[
    "epoch"
] == 8

assert (
    lambda05_development_embeddings.shape
    == (
        904,
        512,
    )
)

assert (
    lambda05_validation_embeddings.shape
    == (
        224,
        512,
    )
)

assert np.bincount(
    lambda05_development_script_labels
).tolist() == [
    452,
    452,
]

assert np.bincount(
    lambda05_validation_script_labels
).tolist() == [
    112,
    112,
]

Best lambda-0.5 epoch: 8
Development embedding shape: (904, 512)
Validation embedding shape: (224, 512)
Development script counts: [452, 452]
Validation script counts: [112, 112]
Development writers: 226
Validation writers: 56


In [18]:
lambda05_script_probe = Pipeline(
    [
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=SEED,
            ),
        ),
    ]
)


lambda05_script_probe.fit(
    lambda05_development_embeddings,
    lambda05_development_script_labels,
)


lambda05_validation_script_probability = (
    lambda05_script_probe
    .predict_proba(
        lambda05_validation_embeddings
    )[
        :,
        1,
    ]
)

lambda05_validation_script_prediction = (
    lambda05_script_probe
    .predict(
        lambda05_validation_embeddings
    )
)


lambda05_script_accuracy = accuracy_score(
    lambda05_validation_script_labels,
    lambda05_validation_script_prediction,
)

lambda05_script_balanced_accuracy = (
    balanced_accuracy_score(
        lambda05_validation_script_labels,
        lambda05_validation_script_prediction,
    )
)

lambda05_script_f1 = f1_score(
    lambda05_validation_script_labels,
    lambda05_validation_script_prediction,
)

lambda05_script_auc = roc_auc_score(
    lambda05_validation_script_labels,
    lambda05_validation_script_probability,
)

lambda05_script_confusion_matrix = (
    confusion_matrix(
        lambda05_validation_script_labels,
        lambda05_validation_script_prediction,
    )
)


HYBRID_SCRIPT_PROBE_AUC = 0.9999

script_auc_reduction = (
    HYBRID_SCRIPT_PROBE_AUC
    - lambda05_script_auc
)


print(
    "Lambda-0.5 writer-disjoint "
    "script probe"
)

print(
    "Accuracy:",
    lambda05_script_accuracy,
)

print(
    "Balanced accuracy:",
    lambda05_script_balanced_accuracy,
)

print(
    "F1:",
    lambda05_script_f1,
)

print(
    "ROC-AUC:",
    lambda05_script_auc,
)

print(
    "Confusion matrix:"
)

print(
    lambda05_script_confusion_matrix
)

print()

print(
    "Previous hybrid script ROC-AUC:",
    HYBRID_SCRIPT_PROBE_AUC,
)

print(
    "Lambda-0.5 script ROC-AUC:",
    lambda05_script_auc,
)

print(
    "Script AUC reduction:",
    script_auc_reduction,
)

print()

print(
    "Writer AUC improvement:",
    lambda05_auc_change,
)


assert (
    0.0
    <= lambda05_script_accuracy
    <= 1.0
)

assert (
    0.0
    <= lambda05_script_auc
    <= 1.0
)

Lambda-0.5 writer-disjoint script probe
Accuracy: 0.9776785714285714
Balanced accuracy: 0.9776785714285714
F1: 0.9775784753363229
ROC-AUC: 0.9985650510204083
Confusion matrix:
[[110   2]
 [  3 109]]

Previous hybrid script ROC-AUC: 0.9999
Lambda-0.5 script ROC-AUC: 0.9985650510204083
Script AUC reduction: 0.0013349489795917435

Writer AUC improvement: 0.028662904555761703


In [20]:
CONDITION_BY_PAGE_PAIR = {
    (1, 2): "arabic_variable_same",
    (3, 4): "english_variable_same",
    (1, 3): "cross_variable_variable",
    (1, 4): "cross_variable_same",
    (2, 3): "cross_same_variable",
    (2, 4): "cross_same_same",
}

WITHIN_SCRIPT_CONDITIONS = [
    "arabic_variable_same",
    "english_variable_same",
]

CROSS_SCRIPT_CONDITIONS = [
    "cross_variable_variable",
    "cross_variable_same",
    "cross_same_variable",
    "cross_same_same",
]


def filename_page_id(
    filename,
):
    return int(
        Path(
            filename
        ).stem.split(
            "_"
        )[-1]
    )


def evaluate_validation_conditions(
    model,
):
    embedding_map = extract_embedding_map(
        model,
        validation_loader,
    )

    evaluation_df = (
        validation_pairs_df
        .copy()
        .reset_index(
            drop=True
        )
    )

    evaluation_df[
        "page_a"
    ] = (
        evaluation_df[
            "filename_a"
        ]
        .map(
            filename_page_id
        )
    )

    evaluation_df[
        "page_b"
    ] = (
        evaluation_df[
            "filename_b"
        ]
        .map(
            filename_page_id
        )
    )

    evaluation_df[
        "condition"
    ] = [
        CONDITION_BY_PAGE_PAIR[
            (
                int(
                    page_a
                ),
                int(
                    page_b
                ),
            )
        ]
        for page_a, page_b in zip(
            evaluation_df[
                "page_a"
            ],
            evaluation_df[
                "page_b"
            ],
        )
    ]

    embeddings_a = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                evaluation_df[
                    "filename_a"
                ]
            )
        ]
    )

    embeddings_b = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                evaluation_df[
                    "filename_b"
                ]
            )
        ]
    )

    evaluation_df[
        "score"
    ] = np.sum(
        embeddings_a
        * embeddings_b,
        axis=1,
    )

    condition_rows = []

    for condition in (
        WITHIN_SCRIPT_CONDITIONS
        + CROSS_SCRIPT_CONDITIONS
    ):
        condition_df = (
            evaluation_df[
                evaluation_df[
                    "condition"
                ] == condition
            ]
        )

        labels = (
            condition_df[
                "pair_label"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        scores = (
            condition_df[
                "score"
            ]
            .to_numpy()
        )

        auc = roc_auc_score(
            labels,
            scores,
        )

        eer, eer_threshold = (
            calculate_canonical_eer(
                labels,
                scores,
            )
        )

        condition_rows.append(
            {
                "condition": condition,
                "pairs": len(
                    condition_df
                ),
                "genuine": int(
                    labels.sum()
                ),
                "impostor": int(
                    (
                        labels
                        == 0
                    ).sum()
                ),
                "auc": float(
                    auc
                ),
                "eer": float(
                    eer
                ),
                "eer_threshold": float(
                    eer_threshold
                ),
            }
        )

    condition_metrics_df = (
        pd.DataFrame(
            condition_rows
        )
    )

    within_auc = float(
        condition_metrics_df[
            condition_metrics_df[
                "condition"
            ].isin(
                WITHIN_SCRIPT_CONDITIONS
            )
        ][
            "auc"
        ]
        .mean()
    )

    cross_auc = float(
        condition_metrics_df[
            condition_metrics_df[
                "condition"
            ].isin(
                CROSS_SCRIPT_CONDITIONS
            )
        ][
            "auc"
        ]
        .mean()
    )

    family_metrics = {
        "within_auc": (
            within_auc
        ),
        "cross_auc": (
            cross_auc
        ),
        "within_minus_cross_gap": (
            within_auc
            - cross_auc
        ),
    }

    return (
        evaluation_df,
        condition_metrics_df,
        family_metrics,
    )


print(
    "Condition-level evaluator ready."
)

Condition-level evaluator ready.


In [21]:
(
    zero_condition_scores_df,
    zero_condition_metrics_df,
    zero_family_metrics,
) = evaluate_validation_conditions(
    best_zero_control_model
)

(
    lambda05_condition_scores_df,
    lambda05_condition_metrics_df,
    lambda05_family_metrics,
) = evaluate_validation_conditions(
    lambda05_best_model
)


condition_comparison_df = (
    zero_condition_metrics_df[
        [
            "condition",
            "pairs",
            "auc",
            "eer",
        ]
    ]
    .rename(
        columns={
            "auc": "zero_auc",
            "eer": "zero_eer",
        }
    )
    .merge(
        lambda05_condition_metrics_df[
            [
                "condition",
                "auc",
                "eer",
            ]
        ].rename(
            columns={
                "auc": "lambda05_auc",
                "eer": "lambda05_eer",
            }
        ),
        on="condition",
        how="inner",
    )
)

condition_comparison_df[
    "auc_change"
] = (
    condition_comparison_df[
        "lambda05_auc"
    ]
    - condition_comparison_df[
        "zero_auc"
    ]
)

condition_comparison_df[
    "eer_change_pp"
] = (
    100.0
    * (
        condition_comparison_df[
            "lambda05_eer"
        ]
        - condition_comparison_df[
            "zero_eer"
        ]
    )
)


print(
    condition_comparison_df[
        [
            "condition",
            "zero_auc",
            "lambda05_auc",
            "auc_change",
            "zero_eer",
            "lambda05_eer",
            "eer_change_pp",
        ]
    ]
    .round(
        {
            "zero_auc": 6,
            "lambda05_auc": 6,
            "auc_change": 6,
            "zero_eer": 6,
            "lambda05_eer": 6,
            "eer_change_pp": 4,
        }
    )
    .to_string(
        index=False
    )
)


print()

print(
    "Zero-control within-script macro AUC:",
    zero_family_metrics[
        "within_auc"
    ],
)

print(
    "Lambda-0.5 within-script macro AUC:",
    lambda05_family_metrics[
        "within_auc"
    ],
)

print(
    "Within-script AUC change:",
    (
        lambda05_family_metrics[
            "within_auc"
        ]
        - zero_family_metrics[
            "within_auc"
        ]
    ),
)

print()

print(
    "Zero-control cross-script macro AUC:",
    zero_family_metrics[
        "cross_auc"
    ],
)

print(
    "Lambda-0.5 cross-script macro AUC:",
    lambda05_family_metrics[
        "cross_auc"
    ],
)

print(
    "Cross-script AUC change:",
    (
        lambda05_family_metrics[
            "cross_auc"
        ]
        - zero_family_metrics[
            "cross_auc"
        ]
    ),
)

print()

print(
    "Zero-control within-minus-cross gap:",
    zero_family_metrics[
        "within_minus_cross_gap"
    ],
)

print(
    "Lambda-0.5 within-minus-cross gap:",
    lambda05_family_metrics[
        "within_minus_cross_gap"
    ],
)

print(
    "Gap change:",
    (
        lambda05_family_metrics[
            "within_minus_cross_gap"
        ]
        - zero_family_metrics[
            "within_minus_cross_gap"
        ]
    ),
)

print()

print(
    "Conditions with AUC improvement:",
    int(
        (
            condition_comparison_df[
                "auc_change"
            ]
            > 0
        ).sum()
    ),
    "/",
    len(
        condition_comparison_df
    ),
)

print(
    "Conditions with EER improvement:",
    int(
        (
            condition_comparison_df[
                "eer_change_pp"
            ]
            < 0
        ).sum()
    ),
    "/",
    len(
        condition_comparison_df
    ),
)


assert len(
    condition_comparison_df
) == 6

assert (
    condition_comparison_df[
        "pairs"
    ] == 3136
).all()

              condition  zero_auc  lambda05_auc  auc_change  zero_eer  lambda05_eer  eer_change_pp
   arabic_variable_same  0.874762      0.864500   -0.010262  0.200649      0.196429        -0.4221
  english_variable_same  0.830038      0.824855   -0.005183  0.239286      0.260390         2.1104
cross_variable_variable  0.730079      0.750974    0.020895  0.339610      0.339286        -0.0325
    cross_variable_same  0.705780      0.688422   -0.017359  0.348052      0.379545         3.1494
    cross_same_variable  0.734868      0.763138    0.028270  0.329870      0.285714        -4.4156
        cross_same_same  0.740184      0.753977    0.013793  0.307792      0.316883         0.9091

Zero-control within-script macro AUC: 0.8524002782931355
Lambda-0.5 within-script macro AUC: 0.8446776437847867
Within-script AUC change: -0.007722634508348758

Zero-control cross-script macro AUC: 0.7277278525046382
Lambda-0.5 cross-script macro AUC: 0.7391277249536178
Cross-script AUC change: 0.01139987

In [22]:
def evaluate_attached_script_head(
    model,
    loader,
):
    model.eval()

    all_labels = []
    all_probabilities = []
    all_predictions = []

    with torch.no_grad():
        for batch in loader:
            images = (
                batch[
                    "image"
                ]
                .to(
                    TRAIN_DEVICE
                )
            )

            labels = (
                batch[
                    "language_label"
                ]
                .long()
                .to(
                    TRAIN_DEVICE
                )
            )

            embeddings = model.encode(
                images
            )

            logits = (
                model
                .script_classifier(
                    embeddings
                )
            )

            probabilities = (
                torch.softmax(
                    logits,
                    dim=1,
                )[
                    :,
                    1,
                ]
            )

            predictions = logits.argmax(
                dim=1
            )

            all_labels.append(
                labels
                .detach()
                .cpu()
                .numpy()
            )

            all_probabilities.append(
                probabilities
                .detach()
                .cpu()
                .numpy()
            )

            all_predictions.append(
                predictions
                .detach()
                .cpu()
                .numpy()
            )

    labels = np.concatenate(
        all_labels
    )

    probabilities = np.concatenate(
        all_probabilities
    )

    predictions = np.concatenate(
        all_predictions
    )

    return {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                predictions,
            )
        ),
        "auc": float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        "confusion_matrix": (
            confusion_matrix(
                labels,
                predictions,
            )
        ),
    }


attached_head_development = (
    evaluate_attached_script_head(
        lambda05_best_model,
        development_probe_loader,
    )
)

attached_head_validation = (
    evaluate_attached_script_head(
        lambda05_best_model,
        validation_loader,
    )
)


print(
    "Attached adversarial head — development"
)

print(
    "Accuracy:",
    attached_head_development[
        "accuracy"
    ],
)

print(
    "Balanced accuracy:",
    attached_head_development[
        "balanced_accuracy"
    ],
)

print(
    "ROC-AUC:",
    attached_head_development[
        "auc"
    ],
)

print(
    "Confusion matrix:"
)

print(
    attached_head_development[
        "confusion_matrix"
    ]
)

print()

print(
    "Attached adversarial head — validation"
)

print(
    "Accuracy:",
    attached_head_validation[
        "accuracy"
    ],
)

print(
    "Balanced accuracy:",
    attached_head_validation[
        "balanced_accuracy"
    ],
)

print(
    "ROC-AUC:",
    attached_head_validation[
        "auc"
    ],
)

print(
    "Confusion matrix:"
)

print(
    attached_head_validation[
        "confusion_matrix"
    ]
)

print()

print(
    "Post-hoc linear-probe validation AUC:",
    lambda05_script_auc,
)

print(
    "Probe minus attached-head AUC:",
    (
        lambda05_script_auc
        - attached_head_validation[
            "auc"
        ]
    ),
)

Attached adversarial head — development
Accuracy: 0.5176991150442478
Balanced accuracy: 0.5176991150442478
ROC-AUC: 0.5087027175189913
Confusion matrix:
[[123 329]
 [107 345]]

Attached adversarial head — validation
Accuracy: 0.47767857142857145
Balanced accuracy: 0.47767857142857145
ROC-AUC: 0.446906887755102
Confusion matrix:
[[  7 105]
 [ 12 100]]

Post-hoc linear-probe validation AUC: 0.9985650510204083
Probe minus attached-head AUC: 0.5516581632653063


In [23]:
FROZEN_HEAD_EPOCHS = 100
FROZEN_HEAD_LR = 1e-3


development_embedding_tensor = torch.tensor(
    lambda05_development_embeddings,
    dtype=torch.float32,
    device=TRAIN_DEVICE,
)

development_label_tensor = torch.tensor(
    lambda05_development_script_labels,
    dtype=torch.long,
    device=TRAIN_DEVICE,
)

validation_embedding_tensor = torch.tensor(
    lambda05_validation_embeddings,
    dtype=torch.float32,
    device=TRAIN_DEVICE,
)

validation_label_tensor = torch.tensor(
    lambda05_validation_script_labels,
    dtype=torch.long,
    device=TRAIN_DEVICE,
)


torch.manual_seed(
    SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        SEED
    )


frozen_embedding_script_head = nn.Linear(
    512,
    2,
).to(
    TRAIN_DEVICE
)


frozen_head_optimizer = torch.optim.AdamW(
    frozen_embedding_script_head.parameters(),
    lr=FROZEN_HEAD_LR,
    weight_decay=WEIGHT_DECAY,
)


for epoch in range(
    1,
    FROZEN_HEAD_EPOCHS + 1,
):
    frozen_embedding_script_head.train()

    frozen_head_optimizer.zero_grad(
        set_to_none=True
    )

    development_logits = (
        frozen_embedding_script_head(
            development_embedding_tensor
        )
    )

    loss = script_criterion(
        development_logits,
        development_label_tensor,
    )

    loss.backward()

    frozen_head_optimizer.step()

    if (
        epoch == 1
        or epoch % 20 == 0
    ):
        with torch.no_grad():
            development_accuracy = (
                development_logits.argmax(
                    dim=1
                )
                == development_label_tensor
            ).float().mean().item()

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss.item():.6f} | "
            f"Development accuracy "
            f"{development_accuracy:.4f}"
        )


frozen_embedding_script_head.eval()

with torch.no_grad():
    validation_logits = (
        frozen_embedding_script_head(
            validation_embedding_tensor
        )
    )

    validation_probabilities = (
        torch.softmax(
            validation_logits,
            dim=1,
        )[
            :,
            1,
        ]
        .detach()
        .cpu()
        .numpy()
    )

    validation_predictions = (
        validation_logits
        .argmax(
            dim=1
        )
        .detach()
        .cpu()
        .numpy()
    )


frozen_head_validation_accuracy = (
    accuracy_score(
        lambda05_validation_script_labels,
        validation_predictions,
    )
)

frozen_head_validation_auc = (
    roc_auc_score(
        lambda05_validation_script_labels,
        validation_probabilities,
    )
)


print()

print(
    "Fresh frozen-embedding head validation accuracy:",
    frozen_head_validation_accuracy,
)

print(
    "Fresh frozen-embedding head validation AUC:",
    frozen_head_validation_auc,
)

print(
    "Joint adversarial-head validation AUC:",
    attached_head_validation[
        "auc"
    ],
)

print(
    "Post-hoc logistic-probe validation AUC:",
    lambda05_script_auc,
)

Epoch 001 | Loss 0.694833 | Development accuracy 0.4812
Epoch 020 | Loss 0.673212 | Development accuracy 0.7190
Epoch 040 | Loss 0.652103 | Development accuracy 0.7865
Epoch 060 | Loss 0.632102 | Development accuracy 0.8319
Epoch 080 | Loss 0.613104 | Development accuracy 0.8684
Epoch 100 | Loss 0.595027 | Development accuracy 0.8905

Fresh frozen-embedding head validation accuracy: 0.90625
Fresh frozen-embedding head validation AUC: 0.9697066326530612
Joint adversarial-head validation AUC: 0.446906887755102
Post-hoc logistic-probe validation AUC: 0.9985650510204083


In [24]:
def train_script_head_only_epoch(
    model,
    epoch,
    optimizer,
):
    model.train()

    script_loader = create_script_loader(
        epoch
    )

    loss_sum = 0.0
    correct = 0
    total = 0

    for batch in script_loader:
        optimizer.zero_grad(
            set_to_none=True
        )

        images = (
            batch[
                "image"
            ]
            .to(
                TRAIN_DEVICE
            )
        )

        labels = (
            batch[
                "language_label"
            ]
            .long()
            .to(
                TRAIN_DEVICE
            )
        )

        with torch.no_grad():
            embeddings = model.encode(
                images
            )

        logits = (
            model
            .script_classifier(
                embeddings
            )
        )

        loss = script_criterion(
            logits,
            labels,
        )

        loss.backward()
        optimizer.step()

        loss_sum += float(
            loss
            .detach()
            .cpu()
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += int(
            (
                predictions
                == labels
            )
            .sum()
            .detach()
            .cpu()
        )

        total += int(
            labels.shape[
                0
            ]
        )

    return {
        "loss": (
            loss_sum
            / len(
                script_loader
            )
        ),
        "accuracy": (
            correct
            / total
        ),
    }


print(
    "Frozen-encoder script-head diagnostic ready."
)

Frozen-encoder script-head diagnostic ready.


In [ ]:
head_only_model = (
    build_fresh_adversarial_model()
)

for parameter in (
    head_only_model
    .backbone
    .parameters()
):
    parameter.requires_grad = False


head_only_optimizer = torch.optim.AdamW(
    head_only_model
    .script_classifier
    .parameters(),
    lr=SCRIPT_HEAD_LR,
    weight_decay=WEIGHT_DECAY,
)


head_only_history = []


for epoch in range(
    1,
    EPOCHS + 1,
):
    training_metrics = (
        train_script_head_only_epoch(
            model=head_only_model,
            epoch=epoch,
            optimizer=(
                head_only_optimizer
            ),
        )
    )

    development_metrics = (
        evaluate_attached_script_head(
            head_only_model,
            development_probe_loader,
        )
    )

    validation_metrics = (
        evaluate_attached_script_head(
            head_only_model,
            validation_loader,
        )
    )

    head_only_history.append(
        {
            "epoch": epoch,
            "training_loss": (
                training_metrics[
                    "loss"
                ]
            ),
            "training_accuracy": (
                training_metrics[
                    "accuracy"
                ]
            ),
            "development_auc": (
                development_metrics[
                    "auc"
                ]
            ),
            "validation_auc": (
                validation_metrics[
                    "auc"
                ]
            ),
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Loss "
        f"{training_metrics['loss']:.6f} | "
        f"Train acc "
        f"{training_metrics['accuracy']:.4f} | "
        f"Dev AUC "
        f"{development_metrics['auc']:.6f} | "
        f"Val AUC "
        f"{validation_metrics['auc']:.6f}"
    )


head_only_history_df = pd.DataFrame(
    head_only_history
)


print()

print(
    "Final development AUC:",
    head_only_history_df.iloc[
        -1
    ][
        "development_auc"
    ],
)

print(
    "Final validation AUC:",
    head_only_history_df.iloc[
        -1
    ][
        "validation_auc"
    ],
)


assert len(
    head_only_history_df
) == EPOCHS

Epoch 01 | Loss 0.656525 | Train acc 0.7942 | Dev AUC 0.994259 | Val AUC 0.999681
